# Bot TDI — Finanzas, política y cultura general de Argentina

## Todo el proyecto corre desde este único notebook

Bot de Telegram que consulta datos públicos de Argentina (dólar, riesgo país, inflación, presidentes históricos) usando la API de [ArgentinaDatos](https://argentinadatos.com/docs/).

**Estado de esta parte (1 de 2):** configuración, registro de interacciones en CSV, `/start` con aviso al admin y arranque del bot.
Las próximas partes suman `/ayuda` con botones, dólar, riesgo país, presidentes, inflación y los mensajes no soportados.

## Cómo funciona el flujo

```text
Persona → Telegram → API de Telegram → polling del notebook → handlers → respuesta
```

- El **kernel de Jupyter es el proceso del bot**: mientras la última celda esté ejecutándose, el bot responde. Si se detiene el kernel, el bot deja de contestar.
- Usamos **polling**: el programa le pregunta a Telegram cada pocos segundos si hay mensajes nuevos.
- Un token solo admite **un proceso haciendo polling**. Si hay dos a la vez (por ejemplo `main.py` y este notebook, o dos compañeros con el mismo token), Telegram devuelve el error 409 y los mensajes se reparten o se pierden.

## Cómo trabajar con este notebook

1. Ejecutá las celdas **en orden, de arriba hacia abajo** (`Shift+Enter`).
2. La **última celda arranca el bot y queda ocupada**; no termina hasta que la interrumpas.
3. Para agregar un handler nuevo: **Interrupt** en la celda de arranque → ejecutá la celda nueva → volvé a ejecutar la celda de arranque.
4. Si cambiás algo de la configuración o del `.env`, reiniciá el kernel (**Restart**) y ejecutá todo de nuevo.

## Dependencias

Se instalan una sola vez en el entorno virtual. La celda queda comentada para que no se reinstale en cada ejecución; si hace falta, sacale el `#` y ejecutala.

In [ ]:
#%pip install pyTelegramBotAPI python-dotenv requests matplotlib

## 1. Configuración y creación del bot

Qué hace esta celda:

- Importa las librerías que usa el notebook.
- Lee `BOT_TOKEN` y `CHAT_ID_ADMIN` desde el archivo `.env` (junto a este notebook). **El token nunca se escribe en el código**, para que no termine en GitHub.
- Si falta el token, corta con un mensaje claro en vez de fallar más adelante.
- Crea el objeto `bot` con `parse_mode="Markdown"`: así `*negrita*` y `_cursiva_` funcionan en todos los mensajes sin repetirlo.
- Define `escapar_md()`, un ayudante para texto que viene de afuera (nombres de usuario, por ejemplo). Si un nombre trae un `_` o un `*`, sin escapar rompería el formato del mensaje.

In [ ]:
import csv
import os
import threading
import time
from datetime import datetime
from pathlib import Path

import telebot
from dotenv import load_dotenv
from telebot import types
from telebot.apihelper import ApiTelegramException

load_dotenv(override=False)
BOT_TOKEN = os.getenv("BOT_TOKEN")
CHAT_ID_ADMIN = os.getenv("CHAT_ID_ADMIN")

if not BOT_TOKEN:
    raise RuntimeError("Falta BOT_TOKEN en el archivo .env (junto a este notebook).")

bot = telebot.TeleBot(BOT_TOKEN, parse_mode="Markdown")


def escapar_md(texto: str) -> str:
    """Escapa los caracteres que rompen el Markdown clásico de Telegram."""
    for c in ("_", "*", "`", "["):
        texto = texto.replace(c, "\\" + c)
    return texto


print("Bot creado. Todavía no está escuchando mensajes.")

## 2. Historial de interacciones en CSV

La consigna pide registrar cada interacción en un `.csv` con estas columnas: **timestamp**, **chat_id**, **nombre** y **mensaje**. Ese archivo después se usa para el comando `/historial` y para el notebook de análisis.

Cómo está armado:

- `registrar_interaccion()` agrega una fila al CSV. Si el archivo no existe, lo crea con el encabezado.
- `descripcion_mensaje()` decide qué texto guardar: el texto del mensaje si lo hay, o una etiqueta (`[audio]`, `[imagen]`, `[video]`, `[documento]`) cuando no es texto.
- `listener_historial()` es un **update listener**: `telebot` lo ejecuta con cada tanda de mensajes nuevos, **antes** de los handlers. Así se registra todo sin repetir código en cada comando.
- El `Lock` evita que dos mensajes simultáneos escriban el archivo al mismo tiempo, porque `telebot` atiende cada mensaje en un hilo distinto.

**Limitación:** el listener recibe solo *mensajes*, no los toques de botones inline. Cuando armemos los botones, los registramos desde su propio handler.

El archivo se guarda en `historial/historial.csv` (relativo a la carpeta del notebook) y está en el `.gitignore`, porque tiene nombres e IDs de usuarios.

In [ ]:
RUTA_CSV = Path("historial") / "historial.csv"
CAMPOS = ["timestamp", "chat_id", "nombre", "mensaje"]
IGNORAR_EN_HISTORIAL = {"/historial", "[botón] historial"}  # no se muestran dentro del propio historial
_lock = threading.Lock()


def registrar_interaccion(chat_id, nombre, mensaje):
    RUTA_CSV.parent.mkdir(exist_ok=True)
    with _lock:
        existe = RUTA_CSV.exists()
        with open(RUTA_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if not existe:
                writer.writerow(CAMPOS)
            writer.writerow(
                [datetime.now().isoformat(timespec="seconds"), chat_id, nombre, mensaje]
            )


def descripcion_mensaje(m):
    if m.text:
        return m.text
    if m.voice or m.audio:
        return "[audio]"
    if m.photo:
        return "[imagen]"
    if m.video or m.video_note:
        return "[video]"
    if m.document:
        return "[documento]"
    return "[otro tipo de mensaje]"


def listener_historial(mensajes):
    for m in mensajes:
        if m.from_user is None:
            continue
        registrar_interaccion(m.chat.id, m.from_user.first_name, descripcion_mensaje(m))


bot.set_update_listener(listener_historial)
print("Registro de historial activado.")

## 3. Comando `/start`

`/start` cumple dos requisitos de la consigna:

- **Saluda al usuario por su nombre** (`message.from_user.first_name`) y lo guía hacia `/ayuda`. El texto usa *negrita* y un emoji.
- **Responde en más de un chat**: además de contestarle a quien escribió, `avisar_al_admin()` le manda un mensaje al chat del administrador (`CHAT_ID_ADMIN` del `.env`) con quién inició el bot.

Detalles:

- El aviso al admin va **después** del saludo y dentro de un `try/except`: si el chat del admin no existe, el ID está mal o el admin nunca habló con el bot, el usuario igual recibe su respuesta.
- Un bot no puede escribirle a alguien que nunca inició una conversación con él, así que el admin tiene que haber hecho `/start` al menos una vez.
- Para ver el aviso en la demo, el `/start` tiene que venir de **otra cuenta**; si lo hacés con la misma cuenta del admin, ves los dos mensajes en el mismo chat.

In [ ]:
def avisar_al_admin(message):
    """Segundo chat de la respuesta: avisa al admin que alguien inició el bot."""
    if not CHAT_ID_ADMIN:
        return
    u = message.from_user
    username = f"@{u.username}" if u.username else "sin username"
    texto = (
        "🔔 *Nuevo /start*\n\n"
        f"Nombre: {escapar_md(u.full_name)}\n"
        f"Usuario: {escapar_md(username)}\n"
        f"Chat ID: `{message.chat.id}`"
    )
    try:
        bot.send_message(int(CHAT_ID_ADMIN), texto)
    except (ApiTelegramException, ValueError):
        pass  # si falla el aviso, que no afecte la respuesta al usuario


@bot.message_handler(commands=["start"])
def comando_start(message):
    nombre = message.from_user.first_name or "che"
    texto = (
        f"¡Hola {nombre}! 👋\n\n"
        "Soy tu bot de *finanzas, política y cultura general* de Argentina.\n"
        "Puedo contarte sobre la cotización del dólar, presidentes históricos, "
        "riesgo país, inflación y más.\n\n"
        "Escribí /ayuda para ver todo lo que puedo hacer."
    )
    bot.reply_to(message, texto)
    avisar_al_admin(message)

## 4. Arrancar el bot (siempre la última celda)

Esta celda **queda ejecutándose** mientras el bot esté activo. Para detenerlo: **Interrupt** (⏹) en la barra de la celda.

Qué hace cada línea:

- `remove_webhook()`: Telegram no permite recibir actualizaciones por webhook y por polling a la vez. Si el bot usó webhook antes, esto lo limpia.
- `skip_pending=True`: ignora los mensajes que se acumularon mientras el bot estaba apagado, para que no responda de golpe una tanda vieja.
- `timeout` y `long_polling_timeout`: cuánto espera cada consulta a Telegram antes de reintentar.
- `allowed_updates=["message", "callback_query"]`: qué tipos de novedades recibe. **`callback_query` es lo que hace funcionar los botones inline**; si se omite, los botones no hacen nada. (El notebook de ejemplo del profe usa solo `message` porque no tenía botones.)

Antes de volver a ejecutar esta celda, interrumpí la anterior o reiniciá el kernel.

In [ ]:
bot.remove_webhook()
time.sleep(1)

print("Bot escuchando con polling. Para detenerlo: Interrupt Kernel.")
bot.infinity_polling(
    skip_pending=True,
    timeout=20,
    long_polling_timeout=20,
    allowed_updates=["message", "callback_query"],
)

## Cómo probar esta parte

1. Ejecutá todas las celdas en orden.
2. Mandale `/start` al bot desde Telegram: tiene que saludarte por tu nombre.
3. Pedile a otra persona que haga `/start`: el aviso te llega a vos (admin).
4. Abrí `historial/historial.csv` y confirmá que cada mensaje agregó una fila con las cuatro columnas.